# Teste isolado — ANA (Notícias, período eleitoral 2026)

Fonte candidata: **ANA - Agência Nacional de Águas e Saneamento Básico**,
setor Saneamento. Notebook **descartável** (Fase 1) — sem dispatcher, sem
`atualizar_status_fonte`, sem gravar nada. Só valida:

1. Scraping da listagem (título, data, categoria, link)
2. Extração do texto completo de uma notícia individual

**Sem filtro de relevância nesta etapa** — a lista real inclui ruído
administrativo (lista telefônica, prêmios) junto com conteúdo regulatório
relevante (consultas públicas, revisão tarifária, agenda regulatória). Isso
é proposital: filtrar por relevância é responsabilidade da etapa de NLP mais
adiante no pipeline, não da captura.

## ⚠️ TODO — URL temporária (período eleitoral 2026)

A URL usada aqui (`.../noticias-e-eventos/noticias-periodo-eleitoral-2026/`)
é **temporária**: a própria página da ANA avisa que esse endereço existe só
por causa da legislação eleitoral de 2026. Depois do fim do período
eleitoral, a ANA volta a publicar no endereço histórico de notícias — que
ainda **não foi identificado**.

**Quando isso quebrar** (a listagem parar de trazer itens novos, ou passar a
devolver 404): procurar o endereço histórico — provavelmente algo como
`gov.br/ana/pt-br/assuntos/noticias-e-eventos/noticias` ou similar — e
atualizar `SITE_URL` abaixo. A estrutura de scraping (`listar_ana`,
`extrair_texto_generico`) deve continuar funcionando sem mudança, já que é
a mesma plataforma Plone/gov.br — só a URL da coleção muda.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import re
import time
import random
import unicodedata
import urllib.parse
from datetime import datetime
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
#
# TODO (pós-período eleitoral 2026): esta URL é temporária -- ver aviso na
# própria página da ANA. Quando o período eleitoral acabar, a ANA volta a
# publicar no endereço histórico (ainda não identificado). Atualizar
# SITE_URL quando isso acontecer -- o resto do scraping deve continuar
# funcionando, é a mesma plataforma Plone/gov.br.
# =============================================================================

SITE_URL = "https://www.gov.br/ana/pt-br/assuntos/noticias-e-eventos/noticias-periodo-eleitoral-2026/"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

TAGS_LIXO = ["script", "style", "noscript", "iframe", "svg", "form", "nav", "header", "footer", "aside", "button"]
SELETORES_CONTEUDO = ["#content-core", "#parent-fieldname-text", "article", "main"]
PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")
PADRAO_DATA_PUBLICADO = re.compile(r"Publicado em\s*(\d{2}/\d{2}/\d{4})")

In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar a listagem (com paginação)

A listagem é renderizada no servidor (Plone), sem JavaScript — mesmo padrão
de ANTT/ANEEL já usados no dispatcher `ingest-scraping`. Cada página tem até
30 itens (`ul.noticias.listagem-noticias-com-foto > li`); a paginação avança
via `?b_start:int=30`, `?b_start:int=60`, etc., até uma página vir vazia.

In [0]:
def listar_ana(max_paginas: int = 5, itens_por_pagina: int = 30) -> list[dict]:
    itens = []

    for pagina in range(max_paginas):
        b_start = pagina * itens_por_pagina
        url_pagina = SITE_URL if b_start == 0 else f"{SITE_URL.rstrip('/')}?b_start:int={b_start}"

        html = baixar_pagina(url_pagina)
        if not html:
            print(f"  -> falha ao baixar página (b_start={b_start}); parando.")
            break

        soup = BeautifulSoup(html, "lxml")
        lista = soup.select_one("ul.noticias.listagem-noticias-com-foto")
        if not lista:
            print(f"  -> nenhuma lista encontrada na página (b_start={b_start}); parando.")
            break

        lis = lista.find_all("li", recursive=False)
        if not lis:
            print(f"  -> página vazia (b_start={b_start}); fim da listagem.")
            break

        for li in lis:
            tag_a = li.select_one("h2.titulo a")
            if not tag_a:
                continue
            categoria = li.select_one("div.categoria-noticia")
            data = li.select_one("span.data")

            data_publicacao = None
            if data:
                texto_data = data.get_text(strip=True)
                m = re.match(r"(\d{2})/(\d{2})/(\d{4})", texto_data)
                if m:
                    dia, mes, ano = m.groups()
                    data_publicacao = f"{ano}-{mes}-{dia}"

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": tag_a["href"].strip(),
                "categoria": categoria.get_text(strip=True) if categoria else None,
                "published_at": data_publicacao,
            })

        print(f"  página b_start={b_start}: {len(lis)} itens.")
        time.sleep(random.uniform(0.5, 1.2))

    return itens

In [0]:
itens = listar_ana()

print(f"\n{len(itens)} notícias listadas.\n")
print(f"{'DATA':<12} {'CATEGORIA':<28} TÍTULO")
print("-" * 110)
for item in itens:
    categoria = (item["categoria"] or "?")[:26]
    print(f"{item['published_at'] or '?':<12} {categoria:<28} {item['titulo'][:60]}")

urls_unicas = {i["url"] for i in itens}
sem_data = [i for i in itens if not i["published_at"]]
sem_categoria = [i for i in itens if not i["categoria"]]

print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"sem data: {len(sem_data)} | sem categoria: {len(sem_categoria)}")
print(f"\nExemplo de link: {itens[0]['url']}")

## Teste 2 — abrir uma notícia e extrair o texto completo

Mesmos seletores já usados em `extrair_texto_generico` do dispatcher
`ingest-scraping` (`#content-core` / `#parent-fieldname-text`) — plataforma
Plone/gov.br, então o padrão se repete.

In [0]:
def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    base = None
    for seletor in SELETORES_CONTEUDO:
        encontrado = soup.select_one(seletor)
        if encontrado and len(encontrado.get_text(strip=True)) > 300:
            base = encontrado
            break
    if base is None:
        base = soup

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def data_publicado_ana(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    m = PADRAO_DATA_PUBLICADO.search(soup.get_text(" ", strip=True))
    if not m:
        return None
    dia, mes, ano = m.group(1).split("/")
    return f"{ano}-{mes}-{dia}"


def extrair_noticia_ana(item: dict) -> Optional[dict]:
    html = baixar_pagina(item["url"])
    if not html:
        return None

    texto = extrair_texto_generico(html)
    data_pagina = data_publicado_ana(html)

    return {
        "titulo": item["titulo"],
        "url": item["url"],
        "categoria": item["categoria"],
        "published_at": data_pagina or item["published_at"],
        "texto": texto,
    }

In [0]:
AMOSTRA = 5

detalhes = []
for item in itens[:AMOSTRA]:
    print(f"\n  [item] {item['titulo'][:90]}")
    detalhe = extrair_noticia_ana(item)
    if detalhe is None:
        print("    -> download falhou.")
        continue
    detalhes.append(detalhe)
    print(f"    -> {len(detalhe['texto'])} chars extraídos.")
    time.sleep(random.uniform(0.5, 1.2))

print(f"\n{len(detalhes)}/{AMOSTRA} notícias abertas com sucesso.")
vazias = [d for d in detalhes if len(d["texto"]) < 200]
print(f"Com texto abaixo de 200 chars: {len(vazias)}")

In [0]:
# Amostra completa da primeira notícia — pra conferir na mão se bate com o
# que aparece no site.
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"CATEGORIA   : {detalhe['categoria']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"])

## Conclusão da Fase 1

Os dois testes passam: listagem paginada extrai título/data/categoria/link
de todos os itens (urls únicas, datas presentes), e o texto completo sai
limpo dos mesmos seletores já usados no dispatcher genérico.

**Avaliação preliminar para a Fase 2**: essa fonte parece encaixar bem no
dispatcher genérico `ingest-scraping` — é a mesma plataforma Plone/gov.br
já usada por ANTT e ANEEL nesse dispatcher, sem necessidade de
Selenium/Playwright ou parsing muito particular. A única particularidade é
a paginação via `b_start:int` (que os dispatchers atuais não implementam
para essas fontes gov.br, mas pode ser adicionada) e a URL temporária do
período eleitoral (ver TODO no topo do notebook).

**Sem filtro de relevância aplicado** — como pedido, a lista completa
(ruído administrativo incluso) é capturada; a filtragem fica para a etapa
de NLP mais adiante no pipeline.